# RAG Pipeline
## CAS NLP 2025/26, University of Bern — Final Project

Full RAG pipeline: retrieval from ChromaDB + generation via LLM.
Local: qwen3:8b via Ollama. Production: gpt-oss-120b via GPUStack.
System prompt: prompts/system_prompt.txt (loaded at runtime).

## 1. Environment Setup

In [1]:
import json
import os
import pathlib
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
import requests

os.chdir('/Users/Olga 1/Documents/01_AD_ASTRA/0102_LANGUAGES/CAS_NLP_2025/final_project')

# Load embedding model
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Load ChromaDB index
chroma_client = chromadb.PersistentClient(path="data/chroma")
collection = chroma_client.get_collection("jugend_medien")

print(f"Collection: {collection.count()} chunks")
print("Ready.")

Collection: 375 chunks
Ready.


## 2. System Prompt and RAG Function

The system prompt instructs the LLM to:
1. Internally reformulate the parent's question using technical vocabulary
   (query expansion at generation level — not pre-retrieval)
2. Answer in the same language as the question
3. Cite the source document for each claim using [source: title]
4. Use simple, accessible language suited for a parent
5. Refer to language-specific official platforms if excerpts are insufficient:
   www.jugendundmedien.ch (DE) / www.jeunesetmedias.ch (FR) / www.giovaniemedia.ch (IT)

### 2.1. Source Title Mapping

Chunk metadata stores filenames (e.g. `jm-b-2024-recommendations_fr.txt`).
For citation in generated responses, filenames are mapped to human-readable
document titles extracted directly from the first lines of each source file —
55 documents covered (15 brochures + 40 flyers).
All titles link implicitly to www.jugendundmedien.ch (DE) / www.jeunesetmedias.ch (FR) / www.giovaniemedia.ch (IT) as the authoritative source.

In [2]:
SOURCE_TITLES = {
    # Brochures
    "jm-b-2024-recommendations_fr.txt": "Compétences numériques — Recommandations pour l'utilisation des médias numériques",
    "jm-b-2024-recommendations_de.txt": "Medienkompetenz — Empfehlungen zum Umgang mit digitalen Medien",
    "jm-b-2024-recommendations_it.txt": "Competenze mediali — Raccomandazioni per l'utilizzo dei media digitali",
    "jm-b-2021-school_fr.txt": "Éducation numérique à l'école",
    "jm-b-2021-school_de.txt": "Medienkompetenz im Schulalltag",
    "jm-b-2021-school_it.txt": "Competenze mediali nella realtà scolastica",
    "jm-b-2022-institutions_fr.txt": "Les compétences numériques dans les institutions d'éducation et de pédagogie spécialisée",
    "jm-b-2022-institutions_de.txt": "Medienkompetenz in sozial-, heil- und sonderpädagogischen Institutionen",
    "jm-b-2022-insitutions_it.txt": "Competenze mediali nelle istituzioni di pedagogia sociale, curativa e speciale",
    "jm-b-2020-radikalisierung_fr.txt": "Discours de prévention de la radicalisation sur Internet",
    "jm-b-2020-radikalisierung_de.txt": "Narrative zur Prävention von Online-Radikalisierung",
    "jm-b-2020-radikalisierung_it.txt": "Narrative per la prevenzione della radicalizzazione in Internet",
    "jm-b-2018-peer-education_fr.txt": "Compétences médiatiques et éducation ou tutorat par les pairs",
    "jm-b-2018-peer-education_de.txt": "Medienkompetenzen und Peer-Education / -Tutoring",
    "jm-b-2016-peer-education_it.txt": "Competenze mediali ed educazione o tutoraggio tra pari",
    # Flyers standard 0-7
    "jm-f-2020-0-7_fr.txt": "Recommandations pour l'utilisation des médias numériques — jusqu'à 7 ans",
    "jm-f-2020-0-7_de.txt": "Empfehlungen für den Umgang mit digitalen Medien — bis 7 Jahre",
    "jm-f-2020-0-7_it.txt": "Raccomandazioni per l'utilizzo dei media digitali — fino ai 7 anni",
    "jm-f-2020-0-7_es.txt": "Recomendaciones sobre el uso de los medios digitales — hasta 7 años",
    "jm-f-2020-0-7_ru.txt": "Рекомендации по обращению с цифровыми медиа — до 7 лет",
    "jm-f-2020-0-7_sq.txt": "Rekomandime për përdorimin e mediave dixhitale — deri në 7 vjeç",
    "jm-f-2020-0-7_ti.txt": "ምኽርታት ንኣተሓሕዛ ምስ ኤለክትሮናዊ ሚድያታት — ክሳብ 7 ዓመት",
    # Flyers standard 6-13
    "jm-f-2020-6-13_fr.txt": "Recommandations pour l'utilisation des médias numériques — 6 à 13 ans",
    "jm-f-2020-6-13_de.txt": "Empfehlungen für den Umgang mit digitalen Medien — 6 bis 13 Jahre",
    "jm-f-2020-6-13_it.txt": "Raccomandazioni per l'utilizzo dei media digitali — 6-13 anni",
    "jm-f-2020-6-13_es.txt": "Recomendaciones sobre el uso de los medios digitales — 6 a 13 años",
    "jm-f-2020-6-13_ru.txt": "Рекомендации по обращению с цифровыми медиа — от 6 до 13 лет",
    "jm-f-2020-6-13_sq.txt": "Rekomandime për përdorimin e mediave dixhitale — midis 6 dhe 13 vjeç",
    "jm-f-2020-6-13_ti.txt": "ምኽርታት ንኣተሓሕዛ ምስ ኤለክትሮናዊ ሚድያታት — ኣብ መንጎ 6ን 13ን",
    # Flyers standard 12-18
    "jm-f-2020-12-18_fr.txt": "Recommandations pour l'utilisation des médias numériques — 12 à 18 ans",
    "jm-f-2020-12-18_de.txt": "Empfehlungen für den Umgang mit digitalen Medien — Jugendliche",
    "jm-f-2020-12-18_it.txt": "Raccomandazioni per l'utilizzo dei media digitali — 12-18 anni",
    "jm-f-2020-12-18_es.txt": "Recomendaciones sobre el uso de los medios digitales — jóvenes",
    "jm-f-2020-12-18_ru.txt": "Рекомендации по обращению с цифровыми медиа — подростков",
    "jm-f-2020-12-18_sq.txt": "Rekomandime për përdorimin e mediave dixhitale — të rinjve",
    "jm-f-2020-12-18_ti.txt": "ምኽርታት ንኣተሓሕዛ ምስ ኤለክትሮናዊ ሚድያታት — መንእሰያት",
    # Easy Language flyers
    "jm-f-easy-2020-0-7_fr.txt": "Internet et médias numériques – Conseils importants — jusqu'à 7 ans",
    "jm-f-easy-2020-0-7_de.txt": "Digitale Medien – Wichtige Empfehlungen — bis 7 Jahre",
    "jm-f-easy-2020-0-7_it.txt": "Media digitali e Internet – Consigli importanti — fino ai 7 anni",
    "jm-f-easy-2020-6-13_fr.txt": "Internet et médias numériques – Conseils importants — 6 à 13 ans",
    "jm-f-easy-2020-6-13_de.txt": "Digitale Medien – Wichtige Empfehlungen — 6 bis 13 Jahre",
    "jm-f-easy-2020-6-13_it.txt": "Media digitali e Internet – Consigli importanti — 6-13 anni",
    "jm-f-easy-2020-12-18_fr.txt": "Internet et médias numériques – Conseils importants — 12 à 18 ans",
    "jm-f-easy-2020-12-18_de.txt": "Digitale Medien – Wichtige Empfehlungen — Jugendliche",
    "jm-f-easy-2020-12-18_it.txt": "Media digitali e Internet – Consigli importanti — 12-18 anni",
    # Image rights flyers
    "jm-f-2021-image-rights-parents_fr.txt": "Que montrez-vous ? 10 étapes pour publier des photos d'enfants de manière plus sûre en ligne",
    "jm-f-2021-image-rights-parents_de.txt": "Was zeigen Sie? 10 Schritte für mehr Sicherheit im Umgang mit Kinderfotos online",
    "jm-f-2021-image-rights-parents_it.txt": "Cosa volete mostrare? 10 passi per pubblicare in modo più sicuro foto di bambini online",
    "jm-f-2021-image-rights-youth_fr.txt": "Que montres-tu ? 10 étapes pour publier des photos de manière plus sûre en ligne",
    "jm-f-2021-image-rights-youth_de.txt": "Was zeigst du? 10 Schritte für mehr Sicherheit im Umgang mit Fotos online",
    "jm-f-2021-image-rights-youth_it.txt": "Cosa vuoi mostrare? 10 passi per pubblicare in modo più sicuro foto online",
    # Extended flyers
    "jm-f-ext_es.txt": "Recomendaciones sobre el uso de los medios digitales (documento extendido)",
    "jm-f-ext_ru.txt": "Рекомендации по обращению с цифровыми медийными средствами (расширенный документ)",
    "jm-f-ext_sq.txt": "Rekomandime për përdorimin e mediave dixhitale (dokument i zgjeruar)",
    "jm-f-ext_ti.txt": "ምኽርታት ንኣተሓሕዛ ምስ ኤለክትሮናውያን ሚድያታት (ሰፊሕ ሰነድ)",
}

print(f"Source titles loaded: {len(SOURCE_TITLES)} documents")

Source titles loaded: 55 documents


### 2.2. System Prompt

Loaded at runtime from prompts/system_prompt.txt.

In [3]:
# Load system prompt from file — modify prompts/system_prompt.txt without touching code--OLD
prompt_path = pathlib.Path("prompts/system_prompt.txt")
prompt_path.parent.mkdir(exist_ok=True)
SYSTEM_PROMPT = prompt_path.read_text(encoding="utf-8")

print(f"System prompt loaded: {len(SYSTEM_PROMPT)} characters")

System prompt loaded: 1078 characters


### 2.3. RAG Function

Core pipeline function: embeds the query, retrieves top-5 chunks
from ChromaDB by cosine similarity, builds context with source titles,
and calls the LLM with the system prompt.

In [5]:
def rag(query, n_results=5):
    # Step 1: encode query using the same embedding model as the index
    query_embedding = model.encode([query])[0]

    # Step 2: retrieve most similar chunks from ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    # Step 3: build context string with human-readable source titles
    context_parts = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        title = SOURCE_TITLES.get(meta["source"], meta["source"])
        context_parts.append(f"[source: {title}]\n{doc[:800]}")
    context = "\n\n---\n\n".join(context_parts)

    # Step 4: build user message combining query and retrieved context
    user_message = f"""Question: {query}

Source excerpts:
{context}

Please answer the question based on the source excerpts above."""

    # Step 5: call Ollama for generation
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen3:8b",
            "prompt": f"{SYSTEM_PROMPT}\n\n{user_message}",
            "stream": False
        }
    )

    return response.json()["response"], results

print("RAG function ready. Top-K:", rag.__defaults__[0])

RAG function ready. Top-K: 5


## 3. RAG Pipeline Tests

Single query tests to validate the full pipeline:
retrieve → build context with source titles → generate response in query language.
Tests cover French baseline (Q01, Q04, Q09), cross-lingual Albanian (Q09/SQ),
Russian Cyrillic (Q01/RU), Spanish control language (Q08/ES),
and Tigrinya Ethiopic/Ge'ez (Q01/TI, generation only — GPU required).

### 3.1. Q01/FR - screentime

In [ ]:
# Q01 FR
answer, results = rag("Mon enfant a 3 ans. Combien de temps par jour pour la télé ou la tablette ?")
print(answer)

La question de la durée d'utilisation de la télévision ou de la tablette pour un enfant de 3 ans n'est pas explicitement répondue dans les extraits fournis. Les documents soulignent que les recommandations dépendent de l'individualisation, des contenus, de la supervision et d'une balance avec d'autres activités (comme le mouvement, les interactions sociales et le sommeil). Aucune limite de temps spécifique n'est indiquée pour cet âge [source: Medienkompetenz — Empfehlungen zum Umgang mit digitalen Medien]. Pour des conseils précis, veuillez consulter www.youthandmedia.ch.


Output  Test Q01 FR
La question de la durée d'utilisation de la télévision ou de la tablette pour un enfant de 3 ans n'est pas explicitement répondue dans les extraits fournis. Les documents soulignent que les recommandations dépendent de l'individualisation, des contenus, de la supervision et d'une balance avec d'autres activités (comme le mouvement, les interactions sociales et le sommeil). Aucune limite de temps spécifique n'est indiquée pour cet âge [source: Medienkompetenz — Empfehlungen zum Umgang mit digitalen Medien]. Pour des conseils précis, veuillez consulter www.youthandmedia.ch.

### 3.2. Q04/FR - cyberbullying

In [ ]:
# Q04 FR
answer, results = rag("On envoie des messages méchants à mon enfant sur WhatsApp ou les réseaux sociaux, encore et encore. Que faire ?")
print(answer)

Si votre enfant reçoit des messages méchants sur WhatsApp ou les réseaux sociaux, il est important de prendre des mesures. Vous pouvez contacter des services de conseil spécialisés dans ces situations, comme **Elternno** ou **Pro Juventute**, qui offrent un soutien gratuit et confidentiel [source: Compétences numériques — Recommandations pour l'utilisation des médias numériques]. Ces services peuvent vous aider à gérer le harcèlement numérique ou les messages inappropriés [source: Competenze mediali — Raccomandazioni per l'utilizzo dei media digitali]. En cas d'urgence, vous pouvez également appeler le **Numéro d'urgence des parents** (0848 35 45 55) [source: Medienkompetenz — Empfehlungen zum Umgang mit digitalen Medien]. N'hésitez pas à solliciter leur aide pour protéger votre enfant et trouver des solutions adaptées.


Output Test Q04 FR
Si votre enfant reçoit des messages méchants sur WhatsApp ou les réseaux sociaux, il est important de prendre des mesures. Vous pouvez contacter des services de conseil spécialisés dans ces situations, comme **Elternno** ou **Pro Juventute**, qui offrent un soutien gratuit et confidentiel [source: Compétences numériques — Recommandations pour l'utilisation des médias numériques]. Ces services peuvent vous aider à gérer le harcèlement numérique ou les messages inappropriés [source: Competenze mediali — Raccomandazioni per l'utilizzo dei media digitali]. En cas d'urgence, vous pouvez également appeler le **Numéro d'urgence des parents** (0848 35 45 55) [source: Medienkompetenz — Empfehlungen zum Umgang mit digitalen Medien]. N'hésitez pas à solliciter leur aide pour protéger votre enfant et trouver des solutions adaptées.

### 3.3. Q09/FR - deepfake 

In [ ]:
# Q09 FR
answer, results = rag("Mon enfant a vu une vidéo bizarre en ligne et il a cru que c'était vrai. Comment lui expliquer que les vidéos peuvent être fausses ?")
print(answer)

Il est important de lui expliquer que les vidéos en ligne ne sont pas toujours vraies. Selon les recommandations, les enfants commencent à comprendre la différence entre la réalité et la fiction vers l’âge de six ou sept ans, mais cela devient plus facile si les parents regardent avec eux et les accompagnent activement [source: Compétences numériques — Recommandations pour l'utilisation des médias numériques]. De plus, les médias ne reflètent pas toujours objectivement la réalité : même les vidéos sérieuses présentent une version partielle ou reconstruite des faits [source: Compétences numériques — Recommandations pour l'utilisation des médias numériques]. Vous pouvez lui montrer que les contenus peuvent être manipulés ou incomplets, et lui apprendre à poser des questions avant de croire tout ce qu’il voit.


Output  Test Q09 FR
Il est important de lui expliquer que les vidéos en ligne ne sont pas toujours vraies. Selon les recommandations, les enfants commencent à comprendre la différence entre la réalité et la fiction vers l’âge de six ou sept ans, mais cela devient plus facile si les parents regardent avec eux et les accompagnent activement [source: Compétences numériques — Recommandations pour l'utilisation des médias numériques]. De plus, les médias ne reflètent pas toujours objectivement la réalité : même les vidéos sérieuses présentent une version partielle ou reconstruite des faits [source: Compétences numériques — Recommandations pour l'utilisation des médias numériques]. Vous pouvez lui montrer que les contenus peuvent être manipulés ou incomplets, et lui apprendre à poser des questions avant de croire tout ce qu’il voit.

### 3.4. Q09/SQ -  deepfake - cross-lingue script latin

In [ ]:
# Q09 SQ
answer, results = rag("Fëmija im pa një video të çuditshme online dhe mendoi se ishte e vërtetë. Si ia shpjegoj që videot mund të jenë të rreme?")
print(answer)

Për të shpjeguar fëmijës që videot mund të jenë të rreme, mund të përdorni këto ide:  

1. **Kohët e mëparshme, fëmijët mund të mësojnë të ndajnë mes realitetit dhe fantazisë**. Në vëllëzërim, fëmijët që janë në mënyrë të vëllëzërim (rreth 6-7 vjeç) mund të mësojnë të ndajnë mes të vërtetë dhe të fikcionit, por këto aftësi zhvillohen me ndihmën e prindërit. Mund të shpjegosh që në këto kohë, fëmijët janë në proces të mësuara të kërkojnë të vërtetë dhe të kontrollojnë të përgjithshëm [source: Compétences numériques — Recommandations pour l'utilisation des médias numériques].  

2. **Videot nuk janë gjithmonë të sakta ose të objektivë**. Mediat, edhe të mësimeve të serioza, nuk janë në gjendje të shfaqin të vërtetën të plotë. Ai që shkruan ose shfaqët një video mund të shikojë nga një këndvështrim të caktuar, të përfshijë vetëm një pjesë të realitetit, ose të përdorë teknika për të ndryshuar të vërtetën [source: Compétences numériques — Recommandations pour l'utilisation des médias numér

Output Test # Q09 SQ — deepfake - cross-lingue script latin
Për të shpjeguar fëmijës që videot mund të jenë të rreme, mund të përdorni këto ide:  

1. **Kohët e mëparshme, fëmijët mund të mësojnë të ndajnë mes realitetit dhe fantazisë**. Në vëllëzërim, fëmijët që janë në mënyrë të vëllëzërim (rreth 6-7 vjeç) mund të mësojnë të ndajnë mes të vërtetë dhe të fikcionit, por këto aftësi zhvillohen me ndihmën e prindërit. Mund të shpjegosh që në këto kohë, fëmijët janë në proces të mësuara të kërkojnë të vërtetë dhe të kontrollojnë të përgjithshëm [source: Compétences numériques — Recommandations pour l'utilisation des médias numériques].  

2. **Videot nuk janë gjithmonë të sakta ose të objektivë**. Mediat, edhe të mësimeve të serioza, nuk janë në gjendje të shfaqin të vërtetën të plotë. Ai që shkruan ose shfaqët një video mund të shikojë nga një këndvështrim të caktuar, të përfshijë vetëm një pjesë të realitetit, ose të përdorë teknika për të ndryshuar të vërtetën [source: Compétences numériques — Recommandations pour l'utilisation des médias numériques].  

3. **Këto teknika (si deepfakes) mund të përdoren për të manipuluar**. Në disa raste, videot mund të shkëmbejë të vërtetën, të shfaqë informacion të padëshiruar, ose të përdorë këndvështrimë për të ndikuar. Mund të shpjegosh që këto janë të rreme dhe që duhet të kërkojnë të verifikojnë informacionin me të tjerë burime [source: Medienkompetenz — Empfehlungen zum Umgang mit digitalen Medien].  

Për të mësuar më sakte, fëmija duhet të mësojë të kontrollojë të përgjithshëm, të përdorë kritikën, dhe të mësojë të mësojnë të përdorë teknologjinë në mënyrë të përshtatshme. Për më tepër informacion, mund të shikoni www.youthandmedia.ch.

### 3.5. Q01/RU screetime - cross-lingue script cyrillic

In [6]:
# Q01 RU — cross-lingual Cyrillic
answer, results = rag("Моему ребёнку 3 года. Сколько можно смотреть телевизор или планшет в день?")
print(answer)

Вопрос о количестве времени, которое ребёнку в возрасте 3 лет можно уделять телевизору или планшету, не охватывается конкретными рекомендациями в предоставлённых источниках. Документы обсуждают использование медиа-устройств детьми в возрасте от 6 лет и старше, а также подчёркивают важность баланса между экранной активностью и физической активностью, а также роль родителей в мониторинге содержания [source: Medienkompetenz — Empfehlungen zum Umgang mit digitalen Medien].  

Для детей 3 лет рекомендуется минимизировать экранное время и приоритизировать игры на свежем воздухе, общение с родителями и развитие навыков через руку. Официальные рекомендации по возрастным нормам можно найти на платформе [www.jugendundmedien.ch](https://www.jugendundmedien.ch) (DE) / [www.jeunesetmedias.ch](https://www.jeunesetmedias.ch) (FR) / [www.giovaniemedia.ch](https://www.giovaniemedia.ch) (IT).


Output Test # Q01 RU — screentime - cross-lingue script cyrillic
Вопрос о количестве времени, которое ребёнку в возрасте 3 лет можно уделять телевизору или планшету, не охватывается конкретными рекомендациями в предоставлённых источниках. Документы обсуждают использование медиа-устройств детьми в возрасте от 6 лет и старше, а также подчёркивают важность баланса между экранной активностью и физической активностью, а также роль родителей в мониторинге содержания [source: Medienkompetenz — Empfehlungen zum Umgang mit digitalen Medien].  

Для детей 3 лет рекомендуется минимизировать экранное время и приоритизировать игры на свежем воздухе, общение с родителями и развитие навыков через руку. Официальные рекомендации по возрастным нормам можно найти на платформе [www.jugendundmedien.ch](https://www.jugendundmedien.ch) (DE) / [www.jeunesetmedias.ch](https://www.jeunesetmedias.ch) (FR) / [www.giovaniemedia.ch](https://www.giovaniemedia.ch) (IT).

### 3.6. Q08 ES - sharenting

In [7]:
# Q08 ES
answer, results = rag("Publico fotos de mi hijo en WhatsApp e Instagram. ¿Es un problema?")
print(answer)

Publicar fotos de su hijo en WhatsApp e Instagram puede plantear riesgos si no se siguen ciertas precauciones. Según las guías oficiales, es importante:  

1. **Verificar las configuraciones de privacidad** para asegurar que solo personas autorizadas puedan ver las fotos [source: Was zeigst du? 10 Schritte für mehr Sicherheit im Umgang mit Kinderfotos online].  
2. **Eliminar datos personales** como el nombre, la fecha de nacimiento o la ubicación, que podrían identificar a su hijo [source: Was zeigst du? 10 Schritte für mehr Sicherheit im Umgang mit Kinderfotos online].  
3. **Reflexionar sobre las consecuencias**: ¿Sería aceptable que otros, incluso los padres, vieran la foto? ¿No haría incomodo a alguien en la imagen? [source: Was zeigst du? 10 Schritte für mehr Sicherheit im Umgang mit Kinderfotos online].  

Si se siguen estos pasos, compartir las fotos puede ser seguro. Sin embargo, si no se toman estas precauciones, podría haber riesgos de exposición inadecuada o uso no autoriza

Output Test # Q08 ES — sharenting 

Publicar fotos de su hijo en WhatsApp e Instagram puede plantear riesgos si no se siguen ciertas precauciones. Según las guías oficiales, es importante:  

1. **Verificar las configuraciones de privacidad** para asegurar que solo personas autorizadas puedan ver las fotos [source: Was zeigst du? 10 Schritte für mehr Sicherheit im Umgang mit Kinderfotos online].  
2. **Eliminar datos personales** como el nombre, la fecha de nacimiento o la ubicación, que podrían identificar a su hijo [source: Was zeigst du? 10 Schritte für mehr Sicherheit im Umgang mit Kinderfotos online].  
3. **Reflexionar sobre las consecuencias**: ¿Sería aceptable que otros, incluso los padres, vieran la foto? ¿No haría incomodo a alguien en la imagen? [source: Was zeigst du? 10 Schritte für mehr Sicherheit im Umgang mit Kinderfotos online].  

Si se siguen estos pasos, compartir las fotos puede ser seguro. Sin embargo, si no se toman estas precauciones, podría haber riesgos de exposición inadecuada o uso no autorizado de la imagen. Para más detalles, consulte la plataforma oficial: [www.jugendundmedien.ch](https://www.jugendundmedien.ch) (DE) / [www.jeunesetmedias.ch](https://www.jeunesetmedias.ch) (FR) / [www.giovaniemedia.ch](https://www.giovaniemedia.ch) (IT).

### 3.7. Note on Tigrinya Generation

Cross-lingual retrieval for Tigrinya queries is confirmed (NB03, NB05).
Generation in Tigrinya requires a stronger multilingual model.
qwen3:8b produces infinite loops or falls back to English when generating
in Tigrinya. Testing on UniBe GPUStack (NB05) confirmed that
Tigrinya generation succeeds with gpt-oss-120b — the failure was
model-specific, not language-specific.

## 4. Infrastructure Constraints

Local development runs on Mac Intel 2020 with PyTorch 2.2.2.
paraphrase-multilingual-MiniLM-L12-v2 is used as the embedding model
throughout the pipeline — for local development and for the GPUStack
deployment alike (see streamlit_app_gpu.py).

qwen3-embedding-0.6b, available via the GPUStack API, was considered
as an alternative production embedding model but was not adopted: it
requires PyTorch 2.4+ (unavailable on this architecture), and switching
would require reindexing the entire corpus — not undertaken, in order
to preserve score consistency across all evaluation results.